# Capstone Phase 1 上机：问题定义与文献综述 -- AI营销Agent系统的因果评估

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实数据/库）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **pydantic** 构建DSR问题定义Schema（问题识别/目标/artifact描述/预期贡献）
2. 用 **arxiv** Python包真实查询arXiv API，获取论文元数据
3. 用 **pandas** 执行PRISMA去重/筛选/纳入四阶段流程
4. 用 **matplotlib** 画PRISMA流程图 + 输出研究问题定义书
5. 理解DeepSeek/RAGAS在LLM辅助文献综述中的应用，天道推演设计研究路径

**真实数据**：arXiv API实时查询 "AI marketing agent" / "causal inference marketing" / "LLM agent marketing" / "AI marketing evaluation"
**整合性**：本Phase整合技能0(Day6研究方法)+技能4(Day1 PRISMA)+模块R(R1 DSR/R4 PRISMA)，为Capstone Phase 2-6奠基


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> arxiv 包需要网络连接访问 arXiv API。pydantic/pandas/matplotlib 离线可用。


In [ ]:
# !pip install arxiv pydantic pandas matplotlib -q
# arxiv 包需要网络连接；pydantic/pandas/matplotlib 离线可用


## 1. 数据背景与营销映射

**Capstone论文方向**：「AI原生化企业的营销智能体系统：从表示工程到因果决策的闭环架构」

**Phase 1目标**：定义研究问题 + 完成系统文献综述，为Phase 2-6奠基

**检索策略**（4条 arXiv 查询）：

| 检索式 | arXiv 查询 | max_results | 用途 |
|--------|-----------|:-----------:|------|
| 检索式1 | `AI marketing agent` | 50 | 核心主题：AI营销Agent |
| 检索式2 | `causal inference marketing` | 40 | 因果推断在营销中的应用 |
| 检索式3 | `LLM agent marketing` | 40 | LLM时代营销Agent |
| 检索式4 | `AI marketing evaluation` | 30 | AI营销效果评估 |

**DSR六步框架**（Hevner 2004; Peffers 2007）：
问题识别 -> 目标定义 -> 设计开发 -> 演示 -> 评估 -> 传播

**PRISMA四步流程**：识别 -> 去重 -> 筛选 -> 纳入

**研究问题**：AI Agent对营销转化的因果效果如何评估？


## 2. 导入依赖

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

print("依赖导入完成：pydantic, pandas, matplotlib, json, time")
print("arxiv 包将在 TODO2 中按需导入")
print("\nCapstone Phase 1: 问题定义与文献综述")
print("整合: 技能0(Day6研究方法) + 技能4(Day1 PRISMA) + 模块R(R1 DSR/R4 PRISMA)")


## TODO 1：用 pydantic 构建 DSR 问题定义 Schema

**目标**：用 pydantic 构建DSR问题定义的结构化Schema，实例化Capstone研究问题定义书。

**DSR问题定义四要素**（对应DSR Step 1-2）：

| 要素 | DSR步骤 | 内容 |
|------|---------|------|
| 问题识别 | Step 1 | 研究背景/现有方案不足/研究重要性 |
| 解决方案目标 | Step 2 | artifact目标/功能性/性能/安全性目标 |
| artifact描述 | Step 2 | artifact类型/核心组件 |
| 预期贡献 | Step 2 | 理论/实践贡献/设计原则 |

**要求**：
1. 定义 ProblemIdentification、SolutionObjectives、ArtifactDescription、ExpectedContributions 四个 BaseModel
2. 定义 ResearchQuestionDefinition 整合以上四要素
3. 实例化为 Capstone 研究问题（AI营销Agent系统的因果评估）
4. 打印研究问题定义书

**提示**：
```python
from pydantic import BaseModel, Field
class ProblemIdentification(BaseModel):
    background: str = Field(..., description="研究背景")
    # ...
```


In [ ]:
# TODO 1：用 pydantic 构建 DSR 问题定义 Schema
# 定义 ProblemIdentification / SolutionObjectives / ArtifactDescription /
#   ExpectedContributions / ResearchQuestionDefinition
# 然后实例化为 Capstone 研究问题（AI营销Agent系统的因果评估）

# ===== 你的代码 =====
class ProblemIdentification(BaseModel):
    # DSR Step 1: 问题识别
    # TODO: 你的代码 - 定义 background/current_limitations/importance 字段


class SolutionObjectives(BaseModel):
    # DSR Step 2: 解决方案目标
    # TODO: 你的代码 - 定义 artifact_goal/functional/performance/safety 字段


class ArtifactDescription(BaseModel):
    # artifact描述
    # TODO: 你的代码 - 定义 artifact_type/core_components 字段


class ExpectedContributions(BaseModel):
    # 预期贡献
    # TODO: 你的代码 - 定义 theoretical/practical/design_principles 字段


class ResearchQuestionDefinition(BaseModel):
    # 研究问题定义书（整合DSR四要素）
    # TODO: 你的代码 - 定义 problem/objectives/artifact/contributions 字段


rq_definition = None  # TODO: 你的代码 - 实例化Capstone研究问题
# ==================

raise NotImplementedError


## TODO 2：用 arxiv 包查询 arXiv API，获取论文元数据

**目标**：用 `arxiv` Python 包查询 arXiv API，获取4个主题的论文元数据。

**要求**：
1. 创建 arxiv.Client（配置 num_retries=5, page_size=50）
2. 对每个查询执行 arxiv.Search（按 Relevance 排序）
3. 提取每篇论文的 title / authors / summary / published / entry_id / primary_category
4. 返回 papers 列表（每篇论文是一个 dict）
5. 每次查询间 sleep 3 秒（arXiv API 速率限制）

**提示**：
```python
import arxiv
client = arxiv.Client(num_retries=5, page_size=50)
search = arxiv.Search(query="AI marketing agent", max_results=50,
                      sort_by=arxiv.SortCriterion.Relevance)
for paper in client.results(search):
    paper.title  # 标题
    paper.summary  # 摘要
    paper.published  # 发表日期
```


In [ ]:
# TODO 2：用 arxiv 包查询 arXiv API，获取论文元数据
# 提示：
#   import arxiv
#   client = arxiv.Client(num_retries=5, page_size=50)
#   queries = [("AI marketing agent", 50), ("causal inference marketing", 40),
#              ("LLM agent marketing", 40), ("AI marketing evaluation", 30)]
#   对每个查询执行 arxiv.Search，提取论文元数据
#   每次查询间 sleep 3 秒（arXiv API 速率限制）

# ===== 你的代码 =====
papers = None  # TODO: 你的代码
# ==================

raise NotImplementedError


## TODO 3：PRISMA 去重（pandas）

**目标**：用 pandas 将论文列表转为 DataFrame，按标题去重，记录去重前后数量。

**PRISMA Step 1 -> Step 2a**：识别阶段获取的论文可能跨查询重复，需按标题去重。

**要求**：
1. 将 papers 列表转为 pandas DataFrame
2. 按标题（小写化）去重，保留首次出现
3. 记录去重前数量（n_identified）和去重后数量（n_after_dedup）
4. 打印去重统计

**提示**：
```python
df = pd.DataFrame(papers)
df['title_lower'] = df['title'].str.lower().str.strip()
df_dedup = df.drop_duplicates(subset='title_lower', keep='first')
```


In [ ]:
# TODO 3：PRISMA 去重（pandas）
# 提示：将 papers 转为 DataFrame，按标题去重

# ===== 你的代码 =====
n_identified = None     # TODO: 你的代码
df_dedup = None         # TODO: 你的代码
n_after_dedup = None    # TODO: 你的代码
# ==================

raise NotImplementedError


## TODO 4：PRISMA 筛选（年份 + AI+营销相关性）

**目标**：用 pandas 执行 PRISMA 筛选，按纳入/排除标准过滤文献。

**纳入标准**：
- 发表年份 >= 2023（确保前沿性）
- 标题或摘要包含AI相关关键词（ai/llm/generative/agent等）
- 标题或摘要包含营销或因果相关关键词（marketing/causal/conversion/evaluation等）

**要求**：
1. 从 published 字段提取年份
2. 定义AI关键词列表和营销/因果关键词列表
3. 筛选：year >= 2023 AND has_ai AND has_marketing
4. 记录筛选后数量（n_screened）

**提示**：
```python
df_dedup['year'] = df_dedup['published'].str[:4].astype(int)
ai_keywords = ['ai', 'artificial intelligence', 'llm', 'agent', ...]
marketing_keywords = ['marketing', 'causal', 'conversion', 'evaluation', ...]
```


In [ ]:
# TODO 4：PRISMA 筛选（年份 + AI+营销相关性）
# 提示：提取年份，定义关键词列表，条件筛选

# ===== 你的代码 =====
df_screened = None    # TODO: 你的代码
n_screened = None     # TODO: 你的代码
# ==================

raise NotImplementedError


## TODO 5：文献研究维度分类 + 研究空白分析

**目标**：构建文献研究维度分类函数，识别2-3个研究空白。

**四大研究维度**（基于标题+摘要的关键词匹配）：

| 维度 | 关键词 | 对应Capstone技能 |
|------|--------|-----------------|
| Agent-Architecture | agent, multi-agent, langgraph, autonomous, tool use | 技能2+5 |
| Causal-Marketing | causal, inference, treatment, ate, uplift, rct | 技能3 |
| LLM-Evaluation | evaluation, benchmark, llm-as-a-judge, deepeval, hallucination | 技能5 |
| Representation-Engineering | embedding, representation, knowledge graph, rag, graphrag | 技能1 |

**要求**：
1. 定义 `classify_dimension(paper)` 函数
2. 对 df_screened 的每篇论文应用分类函数
3. 用 pandas 输出研究维度分布统计
4. 基于分布识别2-3个研究空白（gap analysis）

**提示**：
```python
def classify_dimension(paper):
    text = (paper['title'] + ' ' + paper['summary']).lower()
    if any(k in text for k in ['agent', 'multi-agent', ...]):
        return 'Agent-Architecture'
    # ...
```


In [ ]:
# TODO 5：文献研究维度分类 + 研究空白分析
# 提示：定义 classify_dimension 函数，对每篇论文分类，识别研究空白

# ===== 你的代码 =====
def classify_dimension(paper):
    # TODO: 你的代码
    raise NotImplementedError

df_screened['dimension'] = None  # TODO: 你的代码
dimension_dist = None             # TODO: 你的代码
research_gaps = None              # TODO: 你的代码
# ==================

raise NotImplementedError


## TODO 6：用 matplotlib 画 PRISMA 流程图 + 输出研究问题定义书

**目标**：用 matplotlib 画 PRISMA 流程图（真实数字），并输出完整的研究问题定义书。

**PRISMA 流程图结构**：
```
[识别: n_identified]
       |
[去重: n_after_dedup]  --排除: 重复
       |
[筛选: n_screened]     --排除: 年份/相关性
       |
[纳入: n_included]
```

**要求**：
1. 用 FancyBboxPatch 画方框，FancyArrowPatch 画箭头
2. 每个方框标注阶段名称和论文数
3. 右侧标注排除数量和原因
4. 保存为 prisma_flow.png
5. 输出研究问题定义书（基于TODO1的pydantic模型）

**提示**：
```python
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
```


In [ ]:
# TODO 6：用 matplotlib 画 PRISMA 流程图 + 输出研究问题定义书
# 提示：用 FancyBboxPatch 画方框，FancyArrowPatch 画箭头

# ===== 你的代码 =====
fig = None  # TODO: 你的代码
# ==================

raise NotImplementedError


## 3. 2026前沿：DSR问题识别 + LLM辅助文献综述 + 天道推演

### DSR问题识别在AI系统研究中的应用
DSR（Hevner et al. 2004, MIS Quarterly; Peffers et al. 2007, JMIS）是信息系统的经典研究范式。2026年的趋势是用DSR框架系统化地构建和评估AI Agent系统。Agent系统本身就是一个artifact，其问题识别、目标定义、设计开发、评估方法都是可发表的DSR知识贡献。

### LLM辅助文献综述（DeepSeek/RAGAS）
- **DeepSeek-V3/R1**：开源模型在摘要提取/相关性判断上接近GPT-4，成本1/10
- **RAGAS**：评估LLM生成综述文本的质量（faithfulness/relevancy/precision）
- **应用**：论文摘要自动提取 -> 语义相关性判断 -> 证据合成

### 天道推演设计研究问题路径
用天道推演设计研究问题的路径：
- **沙盘分支1**：Agent因果评估框架（填补"Agent系统缺乏因果验证"空白）
- **沙盘分支2**：表示工程×营销知识图谱（填补"营销数据表示碎片化"空白）
- **沙盘分支3**：人机协作治理（填补"Agent安全治理"空白）

每条分支用**贝叶斯推断**更新概率分布，推演3层（immediate -> near -> far），选择最优研究路径。

> 关键词命中：DSR / DeepSeek / RAGAS / 天道推演 / 多Agent仿真 / 贝叶斯


## 完成检查

完成以上6个TODO后，你应该能：
- [ ] 用pydantic构建DSR问题定义Schema
- [ ] 真实查询arXiv API获取论文元数据
- [ ] 用pandas执行PRISMA去重/筛选/纳入
- [ ] 识别研究空白并分类文献
- [ ] 画PRISMA流程图（真实数字）
- [ ] 输出研究问题定义书

**下一步**：阅读 notes.md 的天道推演部分，用沙盘推演分析你的3条研究路径，选择最优路径进入Phase 2。
